### **Student Name:** Sadegh Samadi
### **Student ID:** 810102588  
### **Course:** Operating Systems - Fall 1404 - CA3


# Part 1 - Thread Interleaving
---

### Experimental Results

The program was executed twice with $N=5$. The observed outputs are as follows:

**Run #1 Output:**
```text
N = 5
I'm thread 0 of 5
I'm thread 2 of 5
I'm thread 1 of 5
I'm thread 3 of 5
I'm thread 4 of 5
```

**Run #2 Output:**
```text
N = 5
I'm thread 1 of 5
I'm thread 3 of 5
I'm thread 2 of 5
I'm thread 0 of 5
I'm thread 4 of 5
```

### Analysis of Results (Question 5)

**1. Unpredictability of Execution Order**
The output shows that the threads do not finish in the order they were created (0, 1, 2, 3, 4). In Run #2, Thread 1 finished first, while Thread 0 finished fourth.
* **Reason:** The Operating System's Scheduler is responsible for deciding which thread runs at any given moment. The `pthread_create` function merely adds the thread to the scheduler's Ready Queue; it does not guarantee immediate execution.

**2. The Impact of `sleep()`**
The `sleep()` function is the primary factor causing the randomness here.
* **Mechanism:** When a thread calls `sleep()`, it moves from the **Running State** to the **Waiting State**. The CPU performs a context switch to run another thread.
* **Wake Up:** Threads wake up after different durations (0 to 3 seconds). A thread that sleeps for 0 seconds (Thread 1 in Run #2 likely did this) will return to the Ready Queue and execute significantly faster than a thread that sleeps for 3 seconds, regardless of which one was created first.

**3. Preemption and Interleaving**
Even without `sleep()`, the output would likely be non-deterministic due to **Preemption**.
* **Time Slicing:** The OS allocates a specific time slice (quantum) to each thread. If Thread 0's time slice expires before it can print, the OS interrupts it and switches to Thread 1. This results in the **interleaving** of instructions, effectively mixing the execution flow of independent threads.



# Part 1 - Array Summation (Single-thread vs Multi-thread)

### Implementation Logic
* **Single-threaded:** A simple `for` loop calculates the sum on the main thread.
* **Multi-threaded:** The array is partitioned into $T$ blocks. $T$ worker threads calculate partial sums, and the main thread aggregates them.
* **Synchronization:** `pthread_join` is used to wait for all threads to finish.

---

### Experimental Results
The following data was collected from three separate runs of the program with $N=10,000,000$.

| Run | Strategy | Threads ($T$) | Time (Seconds) | Speedup vs Single |
| :--- | :--- | :---: | :---: | :---: |
| **1** | Single-threaded | 1 | 0.019121 | - |
| **1** | Multi-threaded | 2 | 0.025625 | **0.74x (Slower)** |
| | | | | |
| **2** | Single-threaded | 1 | 0.019709 | - |
| **2** | Multi-threaded | 4 | 0.024500 | **0.80x (Slower)** |
| | | | | |
| **3** | Single-threaded | 1 | 0.020038 | - |
| **3** | Multi-threaded | 8 | 0.016239 | **1.23x (Faster)** |

---

### Analysis (Questions 4, 5, 6)
**4. Is Multi-threaded always faster?**
No. As seen in the results, for $T=2$ and $T=4$, the multi-threaded approach was actually **slower** than the single-threaded approach.
* **Reason:** Thread creation (`pthread_create`) and synchronization (`pthread_join`) introduce **System Overhead**. When $T=2$ or $T=4$, the time saved by splitting the calculation was not enough to cover the "cost" of managing those extra threads.

**5. The "Break-even" Point**

| Array Size ($N$) | Threads ($T$) | Single-Threaded (s) | Multi-Threaded (s) | Speedup Factor | Result |
| :--- | :---: | :---: | :---: | :---: | :--- |
| **Small ($10^5$)** | 2 | 0.000383 | 0.001543 | 0.25x | **Single is Faster** |
| **Small ($10^5$)** | 4 | 0.000471 | 0.001148 | 0.41x | **Single is Faster** |
| **Small ($10^5$)** | 8 | 0.000451 | 0.000809 | 0.56x | **Single is Faster** |
| | | | | | |
| **Medium ($10^7$)** | 2 | 0.019908 | 0.031471 | 0.63x | **Single is Faster** |
| **Medium ($10^7$)** | 4 | 0.019092 | 0.027866 | 0.69x | **Single is Faster** |
| **Medium ($10^7$)** | 8 | 0.019784 | 0.018360 | 1.08x | **Multi is Faster** |
| | | | | | |
| **Large ($10^8$)** | 2 | 0.197501 | 0.198651 | 0.99x | Tie |
| **Large ($10^8$)** | 4 | 0.190625 | 0.106578 | **1.79x** | **Multi is Faster** |
| **Large ($10^8$)** | 8 | 0.200278 | 0.136811 | **1.46x** | **Multi is Faster** |

The data clearly shows where multi-threading becomes beneficial:
* **For Small Arrays ($10^5$):** The single-threaded approach is consistently faster (up to 4x faster). The task is so small that the overhead of creating threads dominates the execution time.
* **The Boundary ($10^7$):** This is the transition zone. With fewer threads ($T=2, 4$), the overhead still outweighs the gain. However, at $T=8$, the parallelism finally pays off, slightly beating the single-threaded baseline (0.018s vs 0.019s).
* **For Large Arrays ($10^8$):** Multi-threading provides a significant advantage. At $T=4$, the program runs almost **1.8x faster** than the single-threaded version.
* Interestingly, for the largest array, $T=4$ (0.106s) was actually faster than $T=8$ (0.136s). This suggests that adding more threads isn't always better. It could be due to:
    * **Core Saturation:** If the physical CPU does not have 8 true cores, the OS has to time-slice 8 threads onto fewer cores, increasing context-switching overhead.
    * **Memory Bandwidth:** 8 threads trying to read from RAM simultaneously might bottleneck the memory bus.

**6. Granularity and Overhead (Question 6)**
These results illustrate the concept of **Granularity**—the ratio of computation to communication/overhead.
* **Fine-Grained ($10^5$):** The computation per thread is too small. The system spends more time performing system calls (`pthread_create`, context switching) than actually adding numbers. This explains why the multi-threaded code was slower.
* **Coarse-Grained ($10^8$):** Each thread has a massive chunk of work (25 million additions). The fixed cost of creating the thread becomes negligible compared to the time saved by running on multiple cores simultaneously.

**7. Optional**

**8. OS Differences & Granularity Boundaries**

The "Granularity Boundary" (the array size where multi-threading starts beating single-threading) is not a fixed number; it depends heavily on how the Operating System manages threads.

**1. Thread Implementation Model:**
* **Linux (NPTL - Native POSIX Thread Library):** Linux implements threads as "Lightweight Processes" (LWP). The kernel uses the `clone()` system call, which is highly optimized. Creating a thread in Linux is relatively "cheap" and fast.
* **Windows:** Windows treats threads as distinct kernel objects with more complex data structures (Thread Environment Block - TEB, Kernel Stack, etc.). Creating a thread (`CreateThread`) generally involves more overhead than in Linux.

**2. Scheduler Granularity:**
* **Linux (CFS - Completely Fair Scheduler):** Designed to handle massive numbers of threads efficiently with dynamic time-slicing. It can switch contexts very rapidly.
* **Windows Scheduler:** Historically uses a coarser timer tick (quantum). While efficient, the cost of context switching can be slightly higher due to the heavier thread state.

**Conclusion:**
Because Linux threads are "lighter" to create and manage:
* **On Linux:** The break-even point is **lower**. Multi-threading might become profitable at smaller array sizes (e.g., $N=500,000$).
* **On Windows:** The break-even point is **higher**. You might need a larger array (e.g., $N=2,000,000$) to generate enough "work" to justify the higher "cost" of creating the threads.

# Part 1 - Race Condition

### Experimental Results

The program was executed in two modes: **Unsafe (No Mutex)** and **Safe (With Mutex)**.

**Test Case 1: Unsafe Execution (Race Condition)**
* **Parameters:** $M=4$ threads, $K=100,000$ increments per thread.
* **Expected Count:** $4 \times 100,000 = 400,000$.

| Run | Final Counter | Difference (Data Loss) | Result |
| :--- | :---: | :---: | :--- |
| **1** | 194,385 | -205,615 | **FAIL** |
| **2** | 265,829 | -134,171 | **FAIL** |
| **3** | 227,970 | -172,030 | **FAIL** |

*Observation:* Without protection, the final counter is significantly lower than the expected value, and the result changes every time (non-deterministic).

**Test Case 2: Safe Execution (Mutex Enabled)**
* **Parameters:** $M=20$ threads, $K=1,000,000$ increments per thread.
* **Expected Count:** $20 \times 1,000,000 = 20,000,000$.

| Run | Final Counter | Difference | Result |
| :--- | :---: | :---: | :--- |
| **1** | 20,000,000 | 0 | **SUCCESS** |
| **2** | 20,000,000 | 0 | **SUCCESS** |

*Observation:* With the Mutex locked, the program produced the correct result even under a heavy load of 20 million operations.

---

### Analysis (Question 7)

**Q: Why is this situation a "Race Condition"?**

The instruction `global_counter++` appears to be a single line of code, but to the CPU, it is composed of three distinct machine instructions (the **Read-Modify-Write** cycle):
1.  **LOAD:** Read the current value of `global_counter` from memory into a register.
2.  **ADD:** Add 1 to the register.
3.  **STORE:** Write the new value from the register back to memory.

**The Race Scenario:**
Because the threads are running simultaneously (interleaved), the following sequence often occurs:
1.  **Thread A** reads `counter` (value is 50).
2.  **Thread B** reads `counter` (value is **still 50**, because A hasn't saved yet).
3.  **Thread A** increments to 51 and writes 51 to memory.
4.  **Thread B** increments to 51 (from its old read) and writes **51** to memory.

**Result:** Two threads performed increments, but the memory only increased by 1. The update from Thread A was "lost" or overwritten by Thread B. This competition to write to memory is the Race Condition.

---

### Critical Section Analysis (Question 10)

**Q: Where is the Critical Section, and why does lock/unlock solve the problem?**

**1. The Critical Section**
The critical section is the specific part of the code that accesses the shared resource (`global_counter`). In our program, it is:
```c
global_counter++;
```

**2. The Solution (Mutex)** By wrapping this line in pthread_mutex_lock(&lock) and pthread_mutex_unlock(&lock), we enforce Mutual Exclusion.

* **Atomicity**: The lock acts as a gateway. Only one thread can hold the lock at a time.
* **Blocking**: If Thread A holds the lock, and Thread B tries to enter, the OS puts Thread B to sleep. Thread B cannot perform the LOAD step until Thread A has finished the STORE step and released the lock.
* **Outcome**: This forces the operations to happen sequentially (one after another) rather than in parallel, ensuring no updates are lost.

# Part 2: CPU Scheduling Simulator Report

## Results and Analysis
The following tables compare the Average Waiting Time and Average Turnaround Time for the three scenarios.

### Scenario 1: The Standard Mix

| Algorithm | Avg Waiting Time | Avg Turnaround Time |
| :--- | :--- | :--- |
| **FCFS** | 8.25 | 12.25 |
| **SJF** | **3.25** | **7.25** |
| **RR (q=1)** | 6.25 | 10.25 |
| **RR (q=4)** | 7.25 | 11.25 |

**Gantt Charts:**
* **FCFS:** `[P1: 0-8] -> [P2: 8-12] -> [P3: 12-13] -> [P4: 13-16]`
* **SJF:** `[P3: 0-1] -> [P4: 1-4] -> [P2: 4-8] -> [P1: 8-16]`
* **RR (q=4):** `[P1: 0-4] -> [P2: 4-8] -> [P3: 8-9] -> [P4: 9-12] -> [P1: 12-16]`
* **RR (q=1):** `[P1] -> [P2] -> [P3] -> [P4] -> [P1] -> [P2] -> [P4] -> [P1] -> [P2] -> [P4] -> [P1] -> [P2] -> [P1: 12-16]`

**Analysis:**
SJF performed best (Optimality) because it executed the short tasks (P3, P4) immediately. FCFS performed worst because P1 (burst 8) arrived first and delayed all other processes. RR served as a middle ground; with $q=1$, it was more responsive than FCFS but suffered from higher turnaround times than SJF due to the interleaving of the long process P1.

### Scenario 2:

| Algorithm | Avg Waiting Time | Avg Turnaround Time |
| :--- | :--- | :--- |
| **FCFS** | 4.75 | 8.75 |
| **SJF** | **4.00** | **8.00** |
| **RR (q=1)** | 5.50 | 9.50 |
| **RR (q=4)** | 4.50 | 8.50 |

**Gantt Charts:**
* **FCFS:** `[P1: 0-7] -> [P2: 7-11] -> [P3: 11-12] -> [P4: 12-16]`
* **SJF:** `[P1: 0-7] -> [P3: 7-8] -> [P2: 8-12] -> [P4: 12-16]`
* **RR (q=4):** `[P1: 0-4] -> [P2: 4-8] -> [P3: 8-9] -> [P1: 9-12] -> [P4: 12-16]`
* **RR (q=1):** `[P1: 0-2] -> [P2: 2-3] -> [P1: 3-4] -> [P2: 4-5] -> [P3: 5-6] -> [P1: 6-7] -> [P4: 7-8] -> [P2: 8-9] -> [P1: 9-10] -> [P4: 10-11] -> [P2: 11-12] -> [P1: 12-13] -> [P4: 13-14] -> [P1: 14-15] -> [P4: 15-16]`

**Analysis:**
In this scenario, SJF again provided the lowest waiting time. Interestingly, **RR (q=4)** performed better than **RR (q=1)**.

### Scenario 3:

| Algorithm | Avg Waiting Time | Avg Turnaround Time |
| :--- | :--- | :--- |
| **FCFS** | **14.33** | **23.00** |
| **SJF** | **3.00** | **11.67** |
| **RR (q=1)** | 5.67 | 14.33 |
| **RR (q=4)** | 5.67 | 14.33 |

**Gantt Charts (Text Representation):**
* **FCFS:** `[P1: 0-20] -> [P2: 20-23] -> [P3: 23-26]`
* **SJF:** `[P2: 0-3] -> [P3: 3-6] -> [P1: 6-26]`
* **RR (q=4):** `[P1: 0-4] -> [P2: 4-7] -> [P3: 7-10] -> [P1: 10-26]`
* **RR (q=1):** `[P1] -> [P2] -> [P3] -> [P1] -> [P2] -> [P3] -> [P1] -> [P2] -> [P3] -> [P1: 9-26]`

**Analysis:**
This scenario clearly demonstrates the **Convoy Effect** in FCFS, where short processes (P2, P3) get stuck behind a massive process (P1), leading to a average waiting time of 14.33. SJF solved this by running P2 and P3 first. Round Robin successfully mitigated the starvation of P2 and P3 by time-slicing P1, bringing the average wait time down to 5.67, which is much "fairer" than FCFS.

## 4. Discussion Questions

### Q1: Effect of Quantum Size ($q$)
The choice of Time Quantum ($q$) in Round Robin represents a trade-off.
* **Small $q$ (e.g., $q=1$):** Increases the responsiveness of the system. Short processes get CPU time quickly (as seen in Scenario 1 and 3 Gantt charts). However, in a real system, this would increase overhead due to frequent context switching.
* **Large $q$ (e.g., $q=4$):** As $q$ increases, Round Robin behaves more like FCFS. In Scenario 3, since $q=4$ was larger than the burst time of P2 and P3 (3), those processes finished in a single turn once they got the CPU, essentially mimicking FCFS behavior within that specific rotation cycle.

### Q2: Best Algorithm for Delay vs. Fairness
* **Delay (Optimality):** **SJF** is consistently the best algorithm for minimizing Average Waiting Time and Turnaround Time. By prioritizing short jobs, it clears the queue faster.
* **Fairness:** **Round Robin** is the superior algorithm for fairness. In Scenario 3 (FCFS), P2 and P3 starved for 20 units. In RR, they received CPU time almost immediately (within the first few cycles). RR ensures that no process waits indefinitely, regardless of the size of other processes in the queue.

### Q3: Conclusion
For general-purpose systems where process burst times are unknown and interactivity is key, **Round Robin** is the preferred choice due to its fairness and response time. For batch systems where throughput is the priority and burst times are estimable, **SJF** provides the optimal system performance.